In [ ]:
# from datasets import load_dataset

# dataset = load_dataset("unicamp-dl/mmarco", "chinese")["train"]
'''
Downloading builder script: 100%|██████████| 9.96k/9.96k [00:00<00:00, 38.4MB/s]
Downloading readme: 100%|██████████| 3.23k/3.23k [00:00<?, ?B/s]
Repo card metadata block was not found. Setting CardData to empty.
Downloading data: 100%|██████████| 2.72G/2.72G [05:25<00:00, 8.35MB/s]
Downloading data: 100%|██████████| 31.6M/31.6M [00:07<00:00, 4.11MB/s]
Downloading data: 100%|██████████| 905M/905M [01:48<00:00, 8.36MB/s]t]
Downloading data files: 100%|██████████| 3/3 [07:44<00:00, 154.81s/it]
Extracting data files: 100%|██████████| 3/3 [00:00<00:00, 124.58it/s]
Generating train split: 21063999 examples [23:38, 14854.57 examples/s]

样本数太多了 千万级别了  电脑撑不住 暂停了

这加载的是 单语的 样本数据 [query, positive, negative]
并不适合 跨语言的数据
'''


In [ ]:
import ir_datasets
import pandas as pd
import random

In [ ]:
# 从ir_datasets加载 查询语言 和 文档语言 数据
mmarco_v2_zh_train = ir_datasets.load("mmarco/v2/zh/train")
mmarco_v2_ru_train = ir_datasets.load("mmarco/v2/ru/train")

In [ ]:
# 加载 查询语言的 qrels 数据
mmarco_v2_zh_ru_train_5k_qrels_df = pd.DataFrame(mmarco_v2_zh_train.qrels_iter())

In [ ]:
# 随机取出 5000 行数据 
# 使用random_state以确保结果可重复
mmarco_v2_zh_ru_train_5k_qrels_df = mmarco_v2_zh_ru_train_5k_qrels_df.sample(n=5000, random_state=12).reset_index(drop=True)

In [ ]:
# 取出查询数据 dataframe 类型
mmarco_v2_zh_train_queries_df = pd.DataFrame(mmarco_v2_zh_train.queries_iter())
# doc 容器
mmarco_v2_ru_train_docstore = mmarco_v2_ru_train.docs_store()

In [ ]:
mmarco_v2_zh_train_queries_df

In [ ]:
# 将查询文本数据和 qrels 数据进行合并
trip_train_data_df = mmarco_v2_zh_ru_train_5k_qrels_df.merge(
    mmarco_v2_zh_train_queries_df,
    on='query_id',
    how='inner'
)

In [ ]:
# 重命名 query 的 “text” 列名
trip_train_data_df.rename(columns={'text': 'query_text'}, inplace=True)

In [ ]:
# 取出正样本 doc_id
pos_doc_ids = mmarco_v2_zh_ru_train_5k_qrels_df['doc_id'].astype(str).tolist()
# 构建 负样本 doc_id
neg_doc_ids = pos_doc_ids[:]
random.shuffle(neg_doc_ids)
# 确保 每一个正样本 对应的负样本与本身不同
for i in range(len(pos_doc_ids)):
    while neg_doc_ids[i] == pos_doc_ids[i]:
        random.shuffle(neg_doc_ids)

In [ ]:
type(pos_doc_ids)
len(pos_doc_ids)

In [ ]:
# # 构建 pos样本 和 neg样本 列表数据
# docs_train_data_list = []
# for index, row in mmarco_v2_zh_ru_train_5k_qrels_df.iterrows():
#     docs_train_data_list.append([row['query_id'],
#                  row['doc_id'],
#                  mmarco_v2_ru_train_docstore.get(pos_doc_ids[index]).text,
#                  mmarco_v2_ru_train_docstore.get(neg_doc_ids[index]).text
#                  ])

In [ ]:
pos_doc = []
neg_doc = []

for index in range(len(pos_doc_ids)):
    pos_doc.append(mmarco_v2_ru_train_docstore.get(pos_doc_ids[index]).text)
    neg_doc.append(mmarco_v2_ru_train_docstore.get(neg_doc_ids[index]).text)

assert len(pos_doc) == len(pos_doc_ids)
assert len(neg_doc) == len(neg_doc_ids)

trip_train_data_df["positive"] = pos_doc
trip_train_data_df["negative"] = neg_doc

trip_train_data_df.to_csv('./zh-ru-trip-train-data-5k.csv', index=False)


In [ ]:
trip_train_data_df

In [ ]:
# print(mmarco_v2_zh_train_qrels_df)
# print(mmarco_v2_ru_train_qrels_df)
# # 判断两个 qrels 是否完全一样 （id）
# print(mmarco_v2_zh_train_qrels_df.equals(mmarco_v2_ru_train_qrels_df))
# # True

# 看一看两种语言对应的文档数否相同
# print(mmarco_v2_zh_train.docs_count(), mmarco_v2_ru_train.docs_count())

In [ ]:
'''
这个方法进行query和doc的对应合并 太笨了
'''

# train_list = []

# for index, row in mmarco_v2_zh_ru_train_5k_qrels_df.iterrows():
#     query_text = str(mmarco_v2_zh_train_queries_df.loc[mmarco_v2_zh_train_queries_df['query_id'] == row['query_id'], 'text'].iloc[0])
#     doc_text = mmarco_v2_ru_train_docstore.get(str(row['doc_id'])).text
#     train_list.append([query_text, doc_text])

# dataset_train_5k_df = pd.DataFrame(train_list, columns=['queries', 'docs'])

# dataset_train_5k_df
